# MRI_Detection on Google Colab
**Brain-tumour detection in MRI: YOLOv8m baseline vs. YOLOv8m + residual CBAM + weighted multi-scale fusion.**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Waipo97/MRI-Image-Detection/blob/main/MRI_Detection/MRI_Detection_Colab.ipynb)

**Before you start**
1. `Runtime -> Change runtime type -> T4 GPU` (or better).
2. Put the dataset zip (`archive.zip`, Kaggle *Labeled MRI Brain Tumor Dataset*) in your Google Drive at `MyDrive/MRI_Detection/archive.zip`.
3. `Runtime -> Run all`. Everything is saved to `MyDrive/MRI_Detection/runs/`.

**If Colab disconnects**, just run all cells again: finished runs are skipped and an interrupted run resumes from its last epoch.

| Section | What happens |
|---|---|
| 1 Setup | GPU check, mount Drive, clone repo, install |
| 2 Data | copy zip to fast local disk, audit + clean + stratified split, YOLO dataset |
| 3 Baseline | train YOLOv8m (COCO-pretrained) |
| 4 Improved | train YOLOv8m + CBAM + weighted fusion (same config, same split, same seed) |
| 5 Results | val + test tables, per-class, curves, confusion matrices, qualitative examples |
| 6 Optional | ablation, download zip of results |

## 0. Settings (edit here)

In [ ]:
DRIVE_FOLDER = "MRI_Detection"      # folder in MyDrive that holds archive.zip and receives runs/
EPOCHS       = None                 # None = value from configs/train.yaml (150, early stopping patience 40)
REPO_URL     = "https://github.com/Waipo97/MRI-Image-Detection.git"
BRANCH       = "main"
# Optional: download the dataset from Kaggle instead of Drive. Leave "" to use archive.zip from Drive.
# Needs a Kaggle API token (Colab secrets KAGGLE_USERNAME / KAGGLE_KEY, or upload kaggle.json when asked).
KAGGLE_DATASET = ""                 # e.g. "owner/dataset-name"

## 1. Setup

In [ ]:
import subprocess, sys, os
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True, capture_output=True, text=True).stdout or "NO GPU - switch the runtime to GPU")
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, pathlib, subprocess
DRIVE = pathlib.Path("/content/drive/MyDrive") / DRIVE_FOLDER
(DRIVE / "runs").mkdir(parents=True, exist_ok=True)

# code: clone (or update) the repo on local disk
REPO = pathlib.Path("/content/MRI-Image-Detection")
if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO)], check=True)
PROJ = REPO / "MRI_Detection"
os.chdir(PROJ)

# paths used by the library (see src/mri_detection/config.py: get_paths)
WORK = pathlib.Path("/content/mri_work"); WORK.mkdir(exist_ok=True)
os.environ.update(MRI_WORK_DIR=str(WORK), MRI_RUNS_DIR=str(DRIVE / "runs"),
                  MRI_RAW_ZIP=str(WORK / "data/raw/archive.zip"), MRI_WEIGHTS_DIR=str(DRIVE / "weights"))
print("project:", PROJ, "\nruns   :", os.environ["MRI_RUNS_DIR"])

In [ ]:
# Colab already has torch/torchvision. Only the pinned ultralytics (the custom layers hook into its parser) is needed.
!pip -q install ultralytics==8.3.40

## 2. Data

In [ ]:
import shutil
zip_local = pathlib.Path(os.environ["MRI_RAW_ZIP"]); zip_local.parent.mkdir(parents=True, exist_ok=True)
zip_drive = DRIVE / "archive.zip"
if zip_local.exists():
    print("already on local disk")
elif KAGGLE_DATASET:
    try:
        from google.colab import userdata
        os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME"); os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    except Exception:
        from google.colab import files; files.upload()                     # upload kaggle.json
        os.makedirs("/root/.kaggle", exist_ok=True); shutil.move("kaggle.json", "/root/.kaggle/kaggle.json"); os.chmod("/root/.kaggle/kaggle.json", 0o600)
    !pip -q install kaggle
    !kaggle datasets download -d {KAGGLE_DATASET} -p {zip_local.parent} && mv {zip_local.parent}/*.zip {zip_local}
else:
    # 1) expected place, 2) anywhere in your Drive (any .zip whose name contains "archive" or "mri", biggest first)
    found = [zip_drive] if zip_drive.exists() else sorted(
        (p for p in pathlib.Path("/content/drive/MyDrive").rglob("*.zip")
         if any(k in p.name.lower() for k in ("archive", "mri", "brain", "tumor"))),
        key=lambda p: p.stat().st_size, reverse=True)
    if not found:
        raise FileNotFoundError("No dataset zip found in your Drive. Upload it to MyDrive (any folder), then re-run this cell.")
    print("candidates:", *[f"{p} ({p.stat().st_size/1e6:.0f} MB)" for p in found[:5]], sep="\n  ")
    shutil.copy(found[0], zip_local); print("using", found[0])
print(f"{zip_local.stat().st_size/1e6:.0f} MB")

In [ ]:
EP = f"--epochs {EPOCHS}" if EPOCHS else ""
# extract -> audit labels -> drop bad images -> stratified 70/20/10 split (duplicates kept together) -> YOLO folders
!python -u scripts/run.py prepare

In [ ]:
import sys, yaml, random, pandas as pd, matplotlib.pyplot as plt
sys.path.insert(0, str(PROJ / "src"))
import mri_detection as md
from mri_detection import data, viz, models, train, evaluate
cfg = md.load_config()
if EPOCHS: cfg["train"]["epochs"] = EPOCHS
P = md.get_paths(cfg); RUNS, YOLO_DIR, DATA_YAML = P["runs"], P["yolo_dir"], P["data_yaml"]
names = list(yaml.safe_load(open(DATA_YAML))["names"].values()); print("classes:", names)

print("excluded images:"); display(pd.read_csv(P["manifests"] / "excluded_images.csv"))
split = pd.read_csv(P["manifests"] / "split_manifest.csv")
display(pd.crosstab(split.folder, split.split)[["train", "val", "test"]])

random.seed(1)
files = random.sample(sorted((YOLO_DIR / "train" / "images").iterdir()), 8)
imgs = [viz.draw_boxes(f, viz.read_yolo_labels(YOLO_DIR / "train" / "labels" / (f.stem + ".txt")), names) for f in files]
viz.show_grid(imgs, ncols=4, figsize=(14, 3.6)); plt.show()

## 3. Baseline: YOLOv8m
Trained with `configs/train.yaml`, the same file the improved model uses, so any difference is caused by the architecture only.
Progress is saved to Drive after every epoch (`runs/baseline_yolov8m/weights/last.pt`).

In [ ]:
!python -u scripts/run.py baseline {EP}

## 4. Improved model: YOLOv8m + residual CBAM + weighted fusion

In [ ]:
!python -u scripts/run.py improved {EP}

In [ ]:
# Are the new blocks actually used? If every CBAM gamma stays ~0 the attention is effectively switched off.
from ultralytics import YOLO
from mri_detection import modules as cm
m = YOLO(str(RUNS / "proposed_yolov8m" / "weights" / "best.pt")).model
print("CBAM gamma    :", [round(float(b.gamma), 4) for b in m.modules() if isinstance(b, cm.CBAMAttention)])
print("Fusion weights:", [[round(float(w), 3) for w in b.w] for b in m.modules() if isinstance(b, cm.WeightedConcat)])

## 5. Results
Choose conclusions from the **validation** table; the **test** table is the number to report.

In [ ]:
import torch
DEVICE = md.config.resolve_device(cfg["train"]["device"])
res = evaluate.compare(RUNS, DATA_YAML, cfg["train"]["imgsz"], cfg["train"]["batch"], DEVICE)
for sp, (tab, pc) in res.items():
    print(f"\n===== {sp.upper()} ====="); display(tab.round(4)); display(pc.round(4))

In [ ]:
cb, cp = evaluate.load_curve(RUNS, "baseline_yolov8m"), evaluate.load_curve(RUNS, "proposed_yolov8m")
panels = [("train/box_loss", "Train box loss"), ("val/box_loss", "Val box loss"), ("metrics/mAP50(B)", "Val mAP@50"),
          ("metrics/mAP50-95(B)", "Val mAP@50-95"), ("metrics/precision(B)", "Val precision"), ("metrics/recall(B)", "Val recall")]
fig, ax = plt.subplots(2, 3, figsize=(15, 8))
for a, (col, t) in zip(ax.ravel(), panels):
    a.plot(cb["epoch"], cb[col], label="Baseline"); a.plot(cp["epoch"], cp[col], label="Improved")
    a.set_title(t); a.set_xlabel("epoch"); a.grid(alpha=.3); a.legend()
plt.tight_layout(); plt.savefig(RUNS / "training_curves_comparison.png", dpi=150); plt.show()

In [ ]:
from IPython.display import Image as IPImage
for n in ["baseline", "proposed"]:
    f = RUNS / f"{n}_test" / "confusion_matrix_normalized.png"
    print(n, "(test)"); display(IPImage(str(f), width=520)) if f.exists() else print("not found:", f)

In [ ]:
# ground truth | baseline | improved on random TEST images
IMGSZ = cfg["train"]["imgsz"]
test_imgs = sorted((YOLO_DIR / "test" / "images").iterdir())
random.seed(3); picks = random.sample(test_imgs, min(6, len(test_imgs)))
mb = YOLO(str(RUNS / "baseline_yolov8m/weights/best.pt")); mp_ = YOLO(str(RUNS / "proposed_yolov8m/weights/best.pt"))
fig, ax = plt.subplots(len(picks), 3, figsize=(12, 4 * len(picks)), squeeze=False)
for i, p in enumerate(picks):
    gt = viz.draw_boxes(p, viz.read_yolo_labels(YOLO_DIR / "test" / "labels" / (p.stem + ".txt")), names)
    pb = mb.predict(str(p), imgsz=IMGSZ, conf=0.25, verbose=False, device=DEVICE)[0].plot()[..., ::-1]
    pp = mp_.predict(str(p), imgsz=IMGSZ, conf=0.25, verbose=False, device=DEVICE)[0].plot()[..., ::-1]
    for j, (im, t) in enumerate(zip([gt, pb, pp], ["Ground truth", "Baseline", "Improved"])):
        ax[i, j].imshow(im); ax[i, j].set_title(t); ax[i, j].axis("off")
plt.tight_layout(); plt.savefig(RUNS / "qualitative_comparison.png", dpi=130); plt.show()

**How to read it honestly**
- Compare mAP@50-95 and recall first; do not claim an improvement the tables do not show.
- One dataset, one seed, one image-level split (no patient ids, so slices of one patient may sit in different splits). Scores are probably optimistic for new patients.
- Research prototype, **not** a clinical tool.

## 6. Optional
### 6.1 Ablation (attention only / fusion only / both)
Two extra full trainings (each resumable). Set `RUN_ABLATION = True`.

In [ ]:
RUN_ABLATION = False
if RUN_ABLATION:
    train.train_improved(cfg, DATA_YAML, RUNS, "ablation_attention_only", len(names), attention=True, weighted=False)
    train.train_improved(cfg, DATA_YAML, RUNS, "ablation_fusion_only", len(names), attention=False, weighted=True)
    ab = evaluate.compare(RUNS, DATA_YAML, IMGSZ, cfg["train"]["batch"], DEVICE, splits=("val",),
                          runs=(("baseline", "baseline_yolov8m"), ("attention_only", "ablation_attention_only"),
                                ("fusion_only", "ablation_fusion_only"), ("both", "proposed_yolov8m")))
    display(ab["val"][0].round(4)); ab["val"][0].round(4).to_csv(RUNS / "ablation_val.csv")

### 6.2 Download everything (weights + tables + plots)

In [ ]:
out = shutil.make_archive("/content/mri_results", "zip", RUNS, ".")   # runs/ is also already on your Drive
print(out)
from google.colab import files; files.download(out)